# ⚽ Predict the FIFA World Cup 2026

## 📖 Background

The 2026 FIFA World Cup is one of the biggest sporting events in the world, hosted across the United States, Canada, and Mexico. For the first time, the tournament expands to 48 teams, producing 104 matches across the group stage and knockout rounds.

Using machine learning, historical statistics, and soccer domain knowledge, predict match scores, corners, and cards for every fixture. You must submit all your predictions before a single ball is kicked.

The scoring system rewards precision: an exact scoreline earns maximum points, while close predictions still earn partial credit. Later rounds carry score multipliers, so a strong model that holds up in the knockout stages can leapfrog the competition. The challenge is designed to be difficult enough that no one can achieve a perfect score—even with AI assistance—but accessible enough that any data enthusiast can participate and score points.

## 💾 The data

You have access to the following files:

#### `data/group_fixtures.csv` — all 72 group stage matches
| Variable | Description |
|---|---|
| `match_id` | Unique match identifier |
| `group` | Group letter (A–L) |
| `home_team` | Home team name |
| `away_team` | Away team name |
| `date` | Match date (UTC) |
| `venue` | Stadium and city |

#### `data/knockout_slots.csv` — all 32 knockout round slots
| Variable | Description |
|---|---|
| `match_id` | Unique match identifier |
| `round` | Round name (e.g. `Quarter-final`) |
| `multiplier` | Score multiplier for this round |
| `slot_home` | Description of the home team slot (e.g. `Winner Group A`) |
| `slot_away` | Description of the away team slot |

| Variable | Description |
|---|---|

You may also bring in any external data—FIFA rankings, historical match results, player statistics—to build your predictions.

In [1]:
import pandas as pd

group_fixtures = pd.read_csv('data/group_fixtures.csv')
group_fixtures.head()

,match_id,group,home_team,away_team,date_utc,venue
0,1,A,Mexico,South Africa,2026-06-11T19:00:00Z,"Estadio Azteca, Mexico City"
1,2,A,South Korea,UEFA Playoff D,2026-06-12T02:00:00Z,"Estadio Akron, Guadalajara"
2,3,B,Canada,UEFA Playoff A,2026-06-12T19:00:00Z,"BMO Field, Toronto"
3,4,D,USA,Paraguay,2026-06-13T01:00:00Z,"SoFi Stadium, Los Angeles"
4,5,D,Australia,UEFA Playoff C,2026-06-13T04:00:00Z,"BC Place, Vancouver"


In [2]:
knockout_slots = pd.read_csv('data/knockout_slots.csv')
knockout_slots

,match_id,round,multiplier,date_utc,venue,slot_home,slot_away
0,73,Round of 32,1,2026-06-28T19:00:00Z,"SoFi Stadium, Los Angeles",Runner-up Group A,Runner-up Group B
1,74,Round of 32,1,2026-06-29T17:00:00Z,"NRG Stadium, Houston",Winner Group C,Runner-up Group F
2,75,Round of 32,1,2026-06-29T20:30:00Z,"Gillette Stadium, Boston",Winner Group E,Best 3rd (Groups A/B/C/D/F)
3,76,Round of 32,1,2026-06-30T01:00:00Z,"Estadio BBVA, Monterrey",Winner Group F,Runner-up Group C
4,77,Round of 32,1,2026-06-30T17:00:00Z,"AT&T Stadium, Dallas",Runner-up Group E,Runner-up Group I
5,78,Round of 32,1,2026-06-30T21:00:00Z,"MetLife Stadium, East Rutherford",Winner Group I,Best 3rd (Groups C/D/F/G/H)
6,79,Round of 32,1,2026-07-01T01:00:00Z,"Estadio Azteca, Mexico City",Winner Group A,Best 3rd (Groups C/E/F/H/I)
7,80,Round of 32,1,2026-07-01T16:00:00Z,"Mercedes-Benz Stadium, Atlanta",Winner Group L,Best 3rd (Groups E/H/I/J/K)
8,81,Round of 32,1,2026-07-01T20:00:00Z,"Lumen Field, Seattle",Winner Group G,Best 3rd (Groups A/E/H/I/J)
9,82,Round of 32,1,2026-07-02T00:00:00Z,"Levi's Stadium, Santa Clara",Winner Group D,Best 3rd (Groups B/E/F/I/J)


## 💪 Competition challenge

The 2026 World Cup has two phases:

- **Group stage** (matches 1–72): The 48 teams are split into 12 groups of 4. Every team plays the other 3 teams in their group once. The best teams from each group advance to the next phase.
- **Knockout stage** (matches 73–104): Single-elimination rounds — Round of 32, Round of 16, Quarter-finals, Semi-finals, and the Final. Lose once and you're out. Crucially, the two teams playing in each knockout match are not known in advance: they depend on who qualified from the group stage.

Submit predictions for **every match** in both phases. For each match you need to predict:

1. **Score** — the exact final scoreline (e.g. `2-1` means the home team scores 2, the away team scores 1). For knockout matches, the score is the result after 90 minutes and extra time — the penalty shootout is not included.
2. **Corners** — the number of corner kicks awarded in the match
3. **Yellow cards** — the number of yellow cards shown in the match
4. **Red cards** — the number of red cards shown in the match

For **group stage** matches, also predict:
- **Winning team** — which team wins the individual match (use `home`, `away`, or `draw`)

For **knockout round** matches, also predict:
- **Matchup** — which two teams you predict will be playing in that slot. Because the bracket is determined by group stage results, you need to predict which teams advance far enough to meet in each round.
- **Match winner** — which team wins the match (use `home` or `away`)
- **Penalties** — whether the match goes to a penalty shootout (`True` or `False`)

### Scoring system

| Category | Condition | Points |
|---|---|---|
| Score | Exact scoreline | 25 |
| Score | Correct goal difference, wrong score | 10 |
| Score | Correct total goals, wrong score | 10 |
| Corners | Exact number | 10 |
| Corners | Off by 2 | 5 |
| Yellow cards | Exact number | 10 |
| Yellow cards | Off by 1 | 5 |
| Red cards | Exact number | 5 |
| Winning team *(group stage only)* | Correct | 40 |
| Matchup *(knockout only)* | Both teams correct | 20 |
| Matchup *(knockout only)* | One team correct | 10 |
| Match winner *(knockout only)* | Correct | 20 |
| Penalties *(knockout only)* | Correct | 5 |

All points for a match are multiplied by the round factor:

| Round | Multiplier |
|---|---|
| Group stage | ×1 |
| Round of 32 | ×1 |
| Round of 16 | ×2 |
| Quarter-final | ×4 |
| Semi-final | ×8 |
| Third-place playoff | ×8 |
| Final | ×16 |

## 🗓️ Group stage predictions

Fill in your predictions for all 72 group stage matches below.

In [3]:
group_predictions = group_fixtures.copy()

# Fill in your predictions for each match
# Example (match 1 — Mexico vs South Africa): predicted_home_goals=2, predicted_away_goals=1, corners=9, yellow_cards=3, red_cards=0, winning_team='home'
group_predictions['predicted_home_goals'] = None   # e.g. 2
group_predictions['predicted_away_goals'] = None   # e.g. 1
group_predictions['corners']              = None   # e.g. 9
group_predictions['yellow_cards']         = None   # e.g. 3
group_predictions['red_cards']            = None   # e.g. 0
group_predictions['winning_team']         = None   # "home", "away", or "draw"

group_predictions

,match_id,group,home_team,away_team,date_utc,venue,predicted_home_goals,predicted_away_goals,corners,yellow_cards,red_cards,winning_team
0,1,A,Mexico,South Africa,2026-06-11T19:00:00Z,"Estadio Azteca, Mexico City",None,None,None,None,None,None
1,2,A,South Korea,UEFA Playoff D,2026-06-12T02:00:00Z,"Estadio Akron, Guadalajara",None,None,None,None,None,None
2,3,B,Canada,UEFA Playoff A,2026-06-12T19:00:00Z,"BMO Field, Toronto",None,None,None,None,None,None
3,4,D,USA,Paraguay,2026-06-13T01:00:00Z,"SoFi Stadium, Los Angeles",None,None,None,None,None,None
4,5,D,Australia,UEFA Playoff C,2026-06-13T04:00:00Z,"BC Place, Vancouver",None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...
67,68,L,Croatia,Ghana,2026-06-27T21:00:00Z,"Lincoln Financial Field, Philadelphia",None,None,None,None,None,None
68,69,K,Colombia,Portugal,2026-06-27T23:30:00Z,"Hard Rock Stadium, Miami",None,None,None,None,None,None
69,70,K,FIFA Playoff 1,Uzbekistan,2026-06-27T23:30:00Z,"Mercedes-Benz Stadium, Atlanta",None,None,None,None,None,None
70,71,J,Algeria,Austria,2026-06-28T02:00:00Z,"GEHA Field at Arrowhead Stadium, Kansas City",None,None,None,None,None,None


## 🏆 Knockout stage predictions

For knockout matches you also predict **which teams are playing**. Fill in the team names based on your group stage predictions, then add your match predictions.

In [4]:
knockout_predictions = knockout_slots.copy()

# Fill in your predictions for each knockout match
# Example (match 73 — Round of 32): predicted_home_team='Brazil', predicted_away_team='France', predicted_home_goals=1, predicted_away_goals=0, corners=8, yellow_cards=2, red_cards=0, match_winner='home', penalties=False
knockout_predictions['predicted_home_team']  = None   # e.g. "Brazil"
knockout_predictions['predicted_away_team']  = None   # e.g. "France"
knockout_predictions['predicted_home_goals'] = None   # e.g. 1
knockout_predictions['predicted_away_goals'] = None   # e.g. 0
knockout_predictions['corners']              = None   # e.g. 8
knockout_predictions['yellow_cards']         = None   # e.g. 2
knockout_predictions['red_cards']            = None   # e.g. 0
knockout_predictions['match_winner']         = None   # "home" or "away"
knockout_predictions['penalties']            = None   # True or False

knockout_predictions

,match_id,round,multiplier,date_utc,venue,slot_home,slot_away,predicted_home_team,predicted_away_team,predicted_home_goals,predicted_away_goals,corners,yellow_cards,red_cards,match_winner,penalties
0,73,Round of 32,1,2026-06-28T19:00:00Z,"SoFi Stadium, Los Angeles",Runner-up Group A,Runner-up Group B,None,None,None,None,None,None,None,None,None
1,74,Round of 32,1,2026-06-29T17:00:00Z,"NRG Stadium, Houston",Winner Group C,Runner-up Group F,None,None,None,None,None,None,None,None,None
2,75,Round of 32,1,2026-06-29T20:30:00Z,"Gillette Stadium, Boston",Winner Group E,Best 3rd (Groups A/B/C/D/F),None,None,None,None,None,None,None,None,None
3,76,Round of 32,1,2026-06-30T01:00:00Z,"Estadio BBVA, Monterrey",Winner Group F,Runner-up Group C,None,None,None,None,None,None,None,None,None
4,77,Round of 32,1,2026-06-30T17:00:00Z,"AT&T Stadium, Dallas",Runner-up Group E,Runner-up Group I,None,None,None,None,None,None,None,None,None
5,78,Round of 32,1,2026-06-30T21:00:00Z,"MetLife Stadium, East Rutherford",Winner Group I,Best 3rd (Groups C/D/F/G/H),None,None,None,None,None,None,None,None,None
6,79,Round of 32,1,2026-07-01T01:00:00Z,"Estadio Azteca, Mexico City",Winner Group A,Best 3rd (Groups C/E/F/H/I),None,None,None,None,None,None,None,None,None
7,80,Round of 32,1,2026-07-01T16:00:00Z,"Mercedes-Benz Stadium, Atlanta",Winner Group L,Best 3rd (Groups E/H/I/J/K),None,None,None,None,None,None,None,None,None
8,81,Round of 32,1,2026-07-01T20:00:00Z,"Lumen Field, Seattle",Winner Group G,Best 3rd (Groups A/E/H/I/J),None,None,None,None,None,None,None,None,None
9,82,Round of 32,1,2026-07-02T00:00:00Z,"Levi's Stadium, Santa Clara",Winner Group D,Best 3rd (Groups B/E/F/I/J),None,None,None,None,None,None,None,None,None


## ✅ Checklist before publishing into the competition

- Rename your workspace to make it descriptive of your work. N.B. you should leave the notebook name as `notebook.ipynb`.
- Remove redundant cells like the judging criteria, so the workbook is focused on your predictions.
- Make sure all prediction cells are filled in—`None` values will score 0 points.
- Check that all cells run without error.
- Make sure your workbook is published before **June 10, 2026 at 09:00 UTC**.

## ⏳ Time is ticking. Good luck!

In [5]:
import pandas as pd
import numpy as np
import scipy.stats as stats
from collections import defaultdict

np.random.seed(2026)  # Reproducible

group_fixtures = pd.read_csv('data/group_fixtures.csv')
knockout_slots  = pd.read_csv('data/knockout_slots.csv')

print(f'Group fixtures loaded : {len(group_fixtures)} matches')
print(f'Knockout slots loaded : {len(knockout_slots)} matches')

# ── TEAM ATTACK / DEFENSE COEFFICIENTS ──────────────────────────────────────
TEAM_STRENGTHS = {
    'France': {'att': 1.50, 'def': 0.68}, 'Spain': {'att': 1.42, 'def': 0.62},
    'Argentina': {'att': 1.48, 'def': 0.67}, 'England': {'att': 1.38, 'def': 0.70},
    'Portugal': {'att': 1.38, 'def': 0.73}, 'Brazil': {'att': 1.35, 'def': 0.72},
    'Netherlands': {'att': 1.28, 'def': 0.78}, 'Morocco': {'att': 1.08, 'def': 0.72},
    'Belgium': {'att': 1.25, 'def': 0.80}, 'Germany': {'att': 1.28, 'def': 0.82},
    'Croatia': {'att': 1.15, 'def': 0.78}, 'Colombia': {'att': 1.22, 'def': 0.82},
    'Senegal': {'att': 1.08, 'def': 0.88}, 'Mexico': {'att': 1.08, 'def': 0.93},
    'United States': {'att': 1.10, 'def': 0.93}, 'Uruguay': {'att': 1.20, 'def': 0.83},
    'Japan': {'att': 1.15, 'def': 0.88}, 'Switzerland': {'att': 1.12, 'def': 0.85},
    'Austria': {'att': 1.15, 'def': 0.88}, 'Algeria': {'att': 1.05, 'def': 0.93},
    'Turkey': {'att': 1.10, 'def': 0.90}, 'Ecuador': {'att': 1.05, 'def': 0.95},
    'Ivory Coast': {'att': 1.05, 'def': 0.95}, 'Norway': {'att': 1.10, 'def': 0.90},
    'Serbia': {'att': 1.08, 'def': 0.92}, 'Chile': {'att': 1.05, 'def': 0.95},
    'Australia': {'att': 1.00, 'def': 0.98}, 'Tunisia': {'att': 0.95, 'def': 0.98},
    'Paraguay': {'att': 1.00, 'def': 0.97}, 'Korea Republic': {'att': 1.05, 'def': 0.95},
    'Venezuela': {'att': 0.98, 'def': 1.00}, 'Canada': {'att': 1.02, 'def': 0.98},
    'Sweden': {'att': 1.05, 'def': 0.92}, 'Ghana': {'att': 0.95, 'def': 1.02},
    'Romania': {'att': 0.98, 'def': 1.00}, 'South Africa': {'att': 0.88, 'def': 1.05},
    'Iran': {'att': 0.90, 'def': 1.02}, 'Cameroon': {'att': 0.95, 'def': 1.05},
    'Saudi Arabia': {'att': 0.90, 'def': 1.05}, 'Czechia': {'att': 1.05, 'def': 0.93},
    'Bosnia and Herzegovina': {'att': 1.00, 'def': 0.98}, 'Jordan': {'att': 0.82, 'def': 1.10},
    'Uzbekistan': {'att': 0.85, 'def': 1.08}, 'Congo DR': {'att': 0.88, 'def': 1.08},
    'Cape Verde': {'att': 0.82, 'def': 1.10}, 'Iraq': {'att': 0.80, 'def': 1.12},
    'Qatar': {'att': 0.78, 'def': 1.15}, 'Curacao': {'att': 0.72, 'def': 1.20},
    'Panama': {'att': 0.75, 'def': 1.18},
}

DEFAULT_STRENGTH = {'att': 0.88, 'def': 1.08}
AVG_GOALS, AVG_CORNERS, AVG_YELLOW, AVG_RED = 1.35, 9.2, 3.6, 0.15
HOME_BOOST, HOSTS = 1.10, {'United States', 'Mexico', 'Canada'}

# ── ENGINES ──────────────────────────────────────────────────────────────────
def predict_match(home, away):
    h, a = TEAM_STRENGTHS.get(home, DEFAULT_STRENGTH), TEAM_STRENGTHS.get(away, DEFAULT_STRENGTH)
    lam_h = AVG_GOALS * h['att'] * a['def'] * (HOME_BOOST if home in HOSTS else 1)
    lam_a = AVG_GOALS * a['att'] * h['def']
    hg, ag = int(round(lam_h)), int(round(lam_a))
    if hg == ag:
        if lam_h > lam_a + 0.2: hg += 1
        elif lam_a > lam_h + 0.2: ag += 1
    return np.clip(hg, 0, 6), np.clip(ag, 0, 5), int(round(AVG_CORNERS + ((lam_h/(lam_h+lam_a+1e-9))-0.5)*3)), int(np.clip(round(AVG_YELLOW + max(0, 0.5-abs(lam_h-lam_a))), 2, 7)), 0

def monte_carlo_ko_predict(home, away, iterations=10000):
    h, a = TEAM_STRENGTHS.get(home, DEFAULT_STRENGTH), TEAM_STRENGTHS.get(away, DEFAULT_STRENGTH)
    lam_h = AVG_GOALS * h['att'] * a['def'] * (HOME_BOOST if home in HOSTS else 1)
    lam_a = AVG_GOALS * a['att'] * h['def']
    h_sims, a_sims = np.random.poisson(lam_h, iterations), np.random.poisson(lam_a, iterations)
    h_wins, a_wins, pens = 0, 0, 0
    for hg, ag in zip(h_sims, a_sims):
        if hg > ag: h_wins += 1
        elif ag > hg: a_wins += 1
        else:
            if abs(h['att']-a['att']) < 0.05: pens += 1; h_wins += 1 if np.random.rand() > 0.5 else 0; a_wins += 0 if np.random.rand() > 0.5 else 1
            else: h_wins += 1 if h['att'] > a['att'] else 0; a_wins += 1 if a['att'] > h['att'] else 0
    winner = 'home' if h_wins > a_wins else 'away'
    return int(stats.mode(h_sims, keepdims=True).mode[0]), int(stats.mode(a_sims, keepdims=True).mode[0]), 9, 3, 0, winner, (pens/iterations) > 0.2

# ── EXECUTION ────────────────────────────────────────────────────────────────
group_predictions = group_fixtures.copy()
rows = []
for _, row in group_fixtures.iterrows():
    hg, ag, c, y, r = predict_match(row['home_team'], row['away_team'])
    rows.append({'match_id': row['match_id'], 'group': row['group'], 'home_team': row['home_team'], 'away_team': row['away_team'], 'predicted_home_goals': hg, 'predicted_away_goals': ag, 'corners': c, 'yellow_cards': y, 'red_cards': r, 'winning_team': 'home' if hg > ag else ('away' if ag > hg else 'draw')})
group_predictions = pd.DataFrame(rows)

def resolve_group_advancers(preds):
    standings = defaultdict(lambda: {'points':0, 'GD':0, 'GF':0})
    for _, r in preds.iterrows():
        hg, ag = int(r['predicted_home_goals']), int(r['predicted_away_goals'])
        standings[r['home_team']]['GF'] += hg; standings[r['away_team']]['GF'] += ag
        standings[r['home_team']]['GD'] += hg - ag; standings[r['away_team']]['GD'] += ag - hg
        if r['winning_team'] == 'home': standings[r['home_team']]['points'] += 3
        elif r['winning_team'] == 'away': standings[r['away_team']]['points'] += 3
        else: standings[r['home_team']]['points'] += 1; standings[r['away_team']]['points'] += 1
    
    df = pd.DataFrame.from_dict(standings, orient='index').reset_index().rename(columns={'index':'team'})
    # Join with group labels from fixtures
    df = df.merge(preds[['home_team', 'group']].drop_duplicates(), left_on='team', right_on='home_team').drop('home_team', axis=1)
    
    qs, thirds = {}, []
    for grp, gdf in df.groupby('group'):
        gdf = gdf.sort_values(['points','GD','GF'], ascending=False).reset_index(drop=True)
        qs[f'Winner Group {grp}'] = gdf.loc[0, 'team']; qs[f'Runner-up Group {grp}'] = gdf.loc[1, 'team']
        thirds.append(gdf.loc[2].to_dict())
    return qs, pd.DataFrame(thirds).sort_values(['points','GD','GF'], ascending=False)['team'].head(8).tolist()

bracket_mapping, available_thirds = resolve_group_advancers(group_predictions)

def resolve_slot(slot_name, b_map, res_map, thirds_pool):
    if slot_name in b_map: return b_map[slot_name]
    if slot_name in res_map: return res_map[slot_name]
    if '3rd' in slot_name or 'third' in slot_name.lower(): return thirds_pool.pop(0)
    raise KeyError(f"Slot '{slot_name}' not found.")

knockout_predictions = knockout_slots.copy()
resolved_matches = {}
for idx, row in knockout_predictions.iterrows():
    h_team, a_team = resolve_slot(row['slot_home'], bracket_mapping, resolved_matches, available_thirds), resolve_slot(row['slot_away'], bracket_mapping, resolved_matches, available_thirds)
    hg, ag, c, y, r, w, p = monte_carlo_ko_predict(h_team, a_team)
    resolved_matches[f'Winner Match {row["match_id"]}'] = h_team if w=='home' else a_team
    resolved_matches[f'Loser Match {row["match_id"]}'] = a_team if w=='home' else h_team
    knockout_predictions.loc[idx, ['predicted_home_team', 'predicted_away_team', 'predicted_home_goals', 'predicted_away_goals', 'corners', 'yellow_cards', 'red_cards', 'match_winner', 'penalties']] = [h_team, a_team, hg, ag, c, y, r, w, p]

final_match_id = knockout_predictions.iloc[-1]["match_id"]
print(f"Final Champion: {resolved_matches[f'Winner Match {final_match_id}']}")

Group fixtures loaded : 72 matches
Knockout slots loaded : 32 matches
Final Champion: France
